In [ ]:
import pandas as pd
import glob
import os
import sys
import time

from google.colab import drive
drive.mount('/content/drive')

# ── Configuration ─────────────────────────────────────────────────────────────
NTD_ID      = 90199
AGENCY_NAME = 'City of Madera'
YEARS       = list(range(2014, 2025))
OUTPUT_FILE = 'madera_transit_master.csv'
DATA_FOLDER = '/content/drive/My Drive/NTD project/NTD Data'

# ── File pattern matching ─────────────────────────────────────────────────────
def find_file(year, keyword):
    """
    Find NTD file for a given year and keyword.
    Checks year subfolders first, then flat folder structure.
    Case-insensitive on keyword.
    """
    patterns = [
        os.path.join(DATA_FOLDER, str(year), f'{year}*{keyword}*.xlsx'),
        os.path.join(DATA_FOLDER, str(year), f'{year}*{keyword}*.csv'),
        os.path.join(DATA_FOLDER, str(year), f'{year}*{keyword.lower()}*.xlsx'),
        os.path.join(DATA_FOLDER, str(year), f'{year}*{keyword.lower()}*.csv'),
        os.path.join(DATA_FOLDER, str(year), f'{year}*{keyword.replace("_", " ")}*.xlsx'),
        os.path.join(DATA_FOLDER, f'{year}*{keyword}*.xlsx'),
        os.path.join(DATA_FOLDER, f'{year}*{keyword}*.csv'),
        os.path.join(DATA_FOLDER, f'{year}*{keyword.lower()}*.xlsx'),
        os.path.join(DATA_FOLDER, f'{year}*{keyword.lower()}*.csv'),
    ]
    for pattern in patterns:
        matches = glob.glob(pattern)
        if matches:
            return matches[0]
    return None


# ── File loader with full normalization ───────────────────────────────────────
def load_file(path, retries=3):
    if path is None:
        return None

    for attempt in range(retries):
        try:
            if path.endswith('.csv'):
                df = pd.read_csv(path, low_memory=False)
            else:
                # Safe buried header detection
                try:
                    df_check = pd.read_excel(path, header=None, nrows=2)
                    if df_check.shape[0] > 0:
                        non_null_in_row0 = df_check.iloc[0].notna().sum()
                        total_cols = df_check.shape[1]
                        use_header_1 = (total_cols > 0) and (non_null_in_row0 / total_cols < 0.2)
                    else:
                        use_header_1 = False
                except Exception:
                    use_header_1 = False

                if use_header_1:
                    df = pd.read_excel(path, header=1)
                else:
                    df = pd.read_excel(path)

            # rest of normalization stays exactly the same...
            df.columns = df.columns.str.strip()
            df.columns = df.columns.str.replace(r'\s+', ' ', regex=True)
            df.columns = df.columns.str.replace(r'\s*/\s*', '/', regex=True)

            if '5 Digit NTD ID' in df.columns:
                df.rename(columns={'5 Digit NTD ID': 'NTD ID'}, inplace=True)
            if 'Reporter Name' in df.columns and 'Agency Name' not in df.columns:
                df.rename(columns={'Reporter Name': 'Agency Name'}, inplace=True)
            if 'Fares' in df.columns and 'Total Fares' not in df.columns:
                df.rename(columns={'Fares': 'Total Fares'}, inplace=True)

            service_col_map = {
                'Total actual revenue miles':                 'Actual Vehicles/Passenger Car Revenue Miles',
                'Actual Vehicle/Passenger Car Revenue Miles': 'Actual Vehicles/Passenger Car Revenue Miles',
                'Total actual revenue hours':                 'Actual Vehicle/Passenger Car Revenue Hours',
                'Actual Vehicles/Passenger Car Revenue Hours':'Actual Vehicle/Passenger Car Revenue Hours',
                'Vehicles Operated In Maximum Service':       'Vehicles/Passenger Cars Operated in Maximum Service',
            }
            df.rename(columns=service_col_map, inplace=True)

            return df

        except OSError as e:
            if attempt < retries - 1:
                print(f'    Drive read error on attempt {attempt+1}, retrying in 5s... ({e})')
                time.sleep(5)
            else:
                print(f'    ERROR: Could not read {path} after {retries} attempts.')
                raise

# ── Core extraction ───────────────────────────────────────────────────────────
def extract_year(year):
    """
    Extract and calculate all KPIs for a single year.
    Returns a list of row dicts (one per transit mode).
    """


    # ── Locate core files ──────────────────────────────────────────────────
    f_agency   = find_file(year, 'Agency_Information')
    f_expenses = find_file(year, 'Operating_Expenses')
    f_service  = find_file(year, 'Service')
    f_fares    = find_file(year, 'Fare_Revenue')

    missing = [k for k, v in {
        'Agency_Information': f_agency,
        'Operating_Expenses': f_expenses,
        'Service':            f_service,
        'Fare_Revenue':       f_fares,
    }.items() if v is None]

    if missing:
        print(f'  [{year}] WARNING — missing core files: {missing}. Skipping year.')
        return []

    # ── Load core files ────────────────────────────────────────────────────
    agency   = load_file(f_agency)
    expenses = load_file(f_expenses)
    service  = load_file(f_service)
    fares    = load_file(f_fares)


    # ── Safety net: re-apply NTD ID normalization after loading ───────────
    # Catches any file where load_file normalization didn't fire correctly
    for df_obj in [agency, expenses, service, fares]:
        if df_obj is not None:
            if '5 Digit NTD ID' in df_obj.columns:
                df_obj.rename(columns={'5 Digit NTD ID': 'NTD ID'}, inplace=True)
            if 'Reporter Name' in df_obj.columns and 'Agency Name' not in df_obj.columns:
                df_obj.rename(columns={'Reporter Name': 'Agency Name'}, inplace=True)

    # ── Filter to Madera ───────────────────────────────────────────────────
    agency_m   = agency[agency['NTD ID'] == NTD_ID]
    expenses_m = expenses[expenses['NTD ID'] == NTD_ID]
    fares_m    = fares[fares['NTD ID'] == NTD_ID]

    # Service: keep only Annual Total rows
    if 'Time Period' in service.columns:
        service_m = service[
            (service['NTD ID'] == NTD_ID) &
            (service['Time Period'].str.contains('Annual', na=False))
        ]
        if 'Annual Total' in service_m['Time Period'].values:
            service_m = service_m[service_m['Time Period'] == 'Annual Total']
    else:
        service_m = service[service['NTD ID'] == NTD_ID]

    # Fares: operating funds only
    if 'Expense Type' in fares_m.columns:
        fares_ops = fares_m[fares_m['Expense Type'].str.contains('Operations', na=False)]
    else:
        fares_ops = fares_m

    # Service area population
    pop = None
    if len(agency_m) > 0 and 'Service Area Pop' in agency_m.columns:
        pop = agency_m['Service Area Pop'].values[0]

# ── Revenue Sources — funding breakdown (skip 2014, different structure) ──
    fare_ops = federal_ops = state_ops = local_ops = None

    if year != 2014:
        f_rev_sources = find_file(year, 'Revenue_Sources')
        if f_rev_sources:
            # Earlier years have a 'Read Me' sheet as the first sheet
            # so we explicitly find and load the data sheet by name
            try:
                xl = pd.ExcelFile(f_rev_sources)
                data_sheet = next(
                    (s for s in xl.sheet_names if 'revenue' in s.lower() or 'source' in s.lower()),
                    xl.sheet_names[-1]
                )
                rev = pd.read_excel(f_rev_sources, sheet_name=data_sheet)
            except Exception:
                rev = load_file(f_rev_sources)

            # Normalize columns
            rev.columns = rev.columns.str.strip()
            rev.columns = rev.columns.str.replace(r'\s+', ' ', regex=True)
            if '5 Digit NTD ID' in rev.columns:
                rev.rename(columns={'5 Digit NTD ID': 'NTD ID'}, inplace=True)

            rev_m = rev[
                (rev['NTD ID'] == NTD_ID) &
                (rev['Funds Expended Type'].str.contains('Operations', na=False))
            ]

            # Total column name changed between years
            total_col = None
            for candidate in ['Total', 'Total Funds', 'Total of Fares']:
                if candidate in rev_m.columns:
                    total_col = candidate
                    break

            def get_funding(category):
                row = rev_m[rev_m['Funding Category'] == category]
                if len(row) == 0 or total_col is None:
                    return 0.0
                return float(row[total_col].sum())

            fare_ops    = get_funding('Directly Generated')
            federal_ops = get_funding('Federal Government')
            state_ops   = get_funding('State Government')
            local_ops   = get_funding('Local Government')
        else:
            print(f'  [{year}] NOTE — Revenue_Sources not found, funding will be empty.')

    # ── Revenue Vehicle Inventory — average fleet age ──────────────────────
    avg_age_mb = avg_age_dr = None

    f_inventory = find_file(year, 'Revenue_Vehicle_Inventory')
    if f_inventory:
        inv = load_file(f_inventory)
        inv_m = inv[inv['NTD ID'] == NTD_ID].copy()

        # Manufacture Year column name varies slightly
        mfr_col = None
        for candidate in ['Manufacture Year', 'Year Manufactured', 'Mfr Year']:
            if candidate in inv_m.columns:
                mfr_col = candidate
                break

        # Mode column name varies
        mode_col = None
        for candidate in ['Modes', 'Mode', 'Mode/TOS']:
            if candidate in inv_m.columns:
                mode_col = candidate
                break

        if mfr_col and mode_col:
            inv_m['Fleet_Age'] = year - pd.to_numeric(inv_m[mfr_col], errors='coerce')
            avg_age_mb = round(
                inv_m[inv_m[mode_col].str.contains('MB', na=False)]['Fleet_Age'].mean(), 1
            )
            avg_age_dr = round(
                inv_m[inv_m[mode_col].str.contains('DR', na=False)]['Fleet_Age'].mean(), 1
            )
            # Replace NaN with None for clean CSV output
            avg_age_mb = avg_age_mb if pd.notna(avg_age_mb) else None
            avg_age_dr = avg_age_dr if pd.notna(avg_age_dr) else None
        else:
            print(f'  [{year}] NOTE — Could not find Manufacture Year or Mode column in inventory file.')
    else:
        print(f'  [{year}] NOTE — Revenue_Vehicle_Inventory file not found, fleet age will be empty.')

    # ── Extract metrics per mode ───────────────────────────────────────────
    rows = []
    for mode_code, mode_label in [('MB', 'Fixed Route Bus'), ('DR', 'Demand Response')]:
        exp  = expenses_m[expenses_m['Mode'] == mode_code]
        svc  = service_m[service_m['Mode'] == mode_code]
        fare = fares_ops[fares_ops['Mode'] == mode_code] if 'Mode' in fares_ops.columns else fares_ops

        if len(exp) == 0 and len(svc) == 0:
            print(f'  [{year}] No data for mode {mode_code} — skipping.')
            continue

        # Raw metrics
        total_oe = exp['Total Operating Expenses'].sum() if 'Total Operating Expenses' in exp.columns else None

        vrm_col  = 'Actual Vehicles/Passenger Car Revenue Miles'
        vrh_col  = 'Actual Vehicle/Passenger Car Revenue Hours'
        upt_col  = 'Unlinked Passenger Trips (UPT)'
        voms_col = 'Vehicles/Passenger Cars Operated in Maximum Service'

        vrm  = svc[vrm_col].sum()  if vrm_col  in svc.columns else None
        vrh  = svc[vrh_col].sum()  if vrh_col  in svc.columns else None
        upt  = svc[upt_col].sum()  if upt_col  in svc.columns else None
        voms = svc[voms_col].sum() if voms_col in svc.columns else None

        fare_rev = fare['Total Fares'].sum() if 'Total Fares' in fare.columns else 0

        # Debug output if UPT is missing
        if upt is None:
            print(f'  [{year}] WARNING: UPT is None for mode {mode_code}.')
            missing_cols = [c for c in [vrm_col, vrh_col, upt_col, voms_col] if c not in svc.columns]
            if missing_cols:
                print(f'    Missing columns: {missing_cols}')
                print(f'    Available columns: {list(svc.columns)}')

        # KPIs
        oe_per_upt       = round(total_oe / upt, 2)               if (total_oe and upt  and upt  > 0) else None
        upt_per_vrm      = round(upt / vrm, 4)                    if (upt       and vrm  and vrm  > 0) else None
        oe_per_vrm       = round(total_oe / vrm, 4)               if (total_oe  and vrm  and vrm  > 0) else None
        oe_per_vrh       = round(total_oe / vrh, 2)               if (total_oe  and vrh  and vrh  > 0) else None
        farebox_recovery = round(fare_rev / total_oe * 100, 2)    if (total_oe  and total_oe   > 0)    else 0.0

        # Fleet age is mode-specific
        avg_age = avg_age_mb if mode_code == 'MB' else avg_age_dr

        rows.append({
            'Year':                   year,
            'NTD_ID':                 NTD_ID,
            'Agency':                 AGENCY_NAME,
            'Mode':                   mode_label,
            'Mode_Code':              mode_code,
            'Service_Area_Pop':       pop,
            'Operating_Expenses':     total_oe,
            'Fare_Revenue':           fare_rev,
            'VRM':                    vrm,
            'VRH':                    vrh,
            'UPT':                    upt,
            'VOMS':                   voms,
            'OE_per_UPT':             oe_per_upt,
            'OE_per_VRM':             oe_per_vrm,
            'OE_per_VRH':             oe_per_vrh,
            'UPT_per_VRM':            upt_per_vrm,
            'Farebox_Recovery_Pct':   farebox_recovery,
            'Avg_Fleet_Age':          avg_age,
            'Fare_Ops_Funding':       fare_ops,
            'Federal_Ops_Funding':    federal_ops,
            'State_Ops_Funding':      state_ops,
            'Local_Ops_Funding':      local_ops,
        })

    return rows


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f'NTD Extractor — {AGENCY_NAME} (NTD ID {NTD_ID})')
    print(f'Years: {YEARS[0]}–{YEARS[-1]}')
    print(f'Data folder: {DATA_FOLDER}')
    print('-' * 60)

    all_rows = []
    for year in YEARS:
        print(f'Processing {year}...')
        rows = extract_year(year)
        all_rows.extend(rows)
        for r in rows:
            upt_str    = f"{r['UPT']:>8,.0f}"      if r['UPT']        else '     N/A'
            oe_upt_str = f"${r['OE_per_UPT']:>6.2f}" if r['OE_per_UPT'] else '    N/A'
            age_str    = f"{r['Avg_Fleet_Age']:>4.1f} yrs" if r['Avg_Fleet_Age'] else '    N/A'
            print(f"  {r['Mode']:20s} | OE: ${r['Operating_Expenses']:>12,.0f} | "
                  f"UPT: {upt_str} | OE/UPT: {oe_upt_str} | "
                  f"Farebox: {r['Farebox_Recovery_Pct'] or 0:>5.2f}% | "
                  f"Fleet Age: {age_str}")

    if not all_rows:
        print('\nNo data extracted. Check your file names and folder path.')
        sys.exit(1)

    df = pd.DataFrame(all_rows)
    df.to_csv(OUTPUT_FILE, index=False)

    print(f'\n{"=" * 60}')
    print(f'Done. {len(df)} rows written to: {OUTPUT_FILE}')
    print(f'\nColumn summary:')
    print(df.dtypes)
    print(f'\nNull counts:')
    print(df.isnull().sum())
    print(f'\nPreview:')
    print(df.to_string())


if __name__ == '__main__':
    main()

Mounted at /content/drive
NTD Extractor — City of Madera (NTD ID 90199)
Years: 2014–2024
Data folder: /content/drive/My Drive/NTD project/NTD Data
------------------------------------------------------------
Processing 2014...
  Fixed Route Bus      | OE: $     845,828 | UPT:  143,710 | OE/UPT: $  5.89 | Farebox: 12.46% | Fleet Age:  5.3 yrs
  Demand Response      | OE: $     793,120 | UPT:   36,662 | OE/UPT: $ 21.63 | Farebox:  5.53% | Fleet Age:  3.2 yrs
Processing 2015...
  Fixed Route Bus      | OE: $     712,624 | UPT:  131,493 | OE/UPT: $  5.42 | Farebox: 12.81% | Fleet Age:  5.5 yrs
  Demand Response      | OE: $     751,109 | UPT:   40,505 | OE/UPT: $ 18.54 | Farebox:  4.05% | Fleet Age:  4.4 yrs
Processing 2016...
  Fixed Route Bus      | OE: $     846,817 | UPT:  108,391 | OE/UPT: $  7.81 | Farebox: 10.54% | Fleet Age:  7.1 yrs
  Demand Response      | OE: $     901,769 | UPT:   39,146 | OE/UPT: $ 23.04 | Farebox:  3.08% | Fleet Age:  4.4 yrs
Processing 2017...
  Fixed Route 